# Database Generation

In [2]:
import numpy as np
np.infty = np.inf
import sys
import numpy as np
import cantera as ct
import matplotlib.pyplot as plt
import itertools
import scipy.integrate
import ctypes as xt
import os
from ideal_reactor_models import customESC_BM
from ideal_reactor_models import customPFR
import pandas as pd

In [4]:
def calc_conversion(gas,states, reactant):
    return 100.*(states.Y[0,gas.species_index(reactant)]-states.Y[:,gas.species_index(reactant)])/states.Y[0,gas.species_index(reactant)]
def calc_yield(gas,states, reactant, product):
    return 100.*(states.Y[:,gas.species_index(product)]-states.Y[0,gas.species_index(product)])/states.Y[0,gas.species_index(reactant)]
def calc_selectivity(gas,states, reactant, product):
    return 100.*(states.Y[:,gas.species_index(product)]-states.Y[0,gas.species_index(product)])/(states.Y[0,gas.species_index(reactant)]-states.Y[:,gas.species_index(reactant)])

In [6]:
def CRACKSIM_rates_DLL(gas):
    # CRACKSIM_rates_C call the rates function in CRACKSIM.DLL
    # concentration bassis 

    # initialize temperature and concentrations
    T= gas.T     # [K]
    C_point=gas.concentrations #[mol/l]
    status = 0              

    # copy values to Ctypes to be used as arguements for the Fortran DLL
    C_point= gas.concentrations.ctypes
    T_point = xt.byref(xt.c_double(T)) 
    status = xt.pointer(xt.c_int(status))


    # initalize a ctype pointer to be used as storage for the calculated rates 
    R_point = (xt.c_double*gas.n_species)()     # mol/(s.L)
  
    _ = fortlib.NetRates_C(T_point,C_point,R_point,status)  # function call of CRACKSIM DLL
    
    # convert Ctype to python array
    rates=np.ctypeslib.as_array(R_point)
    rates=rates         #[mol/l/s]
    return rates
    #return rates

In [8]:
# initialize kinetics 
fortlib = xt.CDLL(r"C:\Users\Louis\OneDrive\Bureaublad\ugent\Master2\Thesis\Code_Turboreactor\SA_CRACKSIM.dll") #change this to the location where you stored the dll
status = 0
option = (xt.c_int*20)()
# please note that in python arrays possitions starts counting from 0 
            # option(1) : 
            #      0 use full network
            #      1 use reduced network based on compossition file 
            #      2 use only betanetwork
option[0]= 1
status = xt.pointer(xt.c_int(status)) # setup the pointer to the correct data structure
_ = fortlib.Initialise_CRACKSIM(status,option)            # call the function
status = status[0]
if status==1:
    print("Kinetics and Thermo were read succesfuly")
else:
    print("Errors while reading Kinetics and Thermo")

Kinetics and Thermo were read succesfuly


## Foutmelding NASA polynomes

In [11]:
#Included to suppress the error warnings related to the NASA polynomes
ct.suppress_thermo_warnings()       # currently an issue with Nasapolynomials 
# convert the chem.inp file created by the DLL to yaml file

!ck2yaml --input=chem.inp --transport=transport_chemkin.DAT --permissive > C2KYAML_log.txt

C:\Users\Louis\anacondadownload\envs\cantera-env\lib\site-packages\cantera\ck2yaml.py:2344: UserWarning: NasaPoly2::validate: 
For species CO, discontinuity in h/RT detected at Tmid = 1400
	Value computed using low-temperature polynomial:  -6.48615241904762
	Value computed using high-temperature polynomial: -6.431806107428571

  gas = Solution(out_name)
C:\Users\Louis\anacondadownload\envs\cantera-env\lib\site-packages\cantera\ck2yaml.py:2344: UserWarning: NasaPoly2::validate: 
For species CO2, discontinuity in h/RT detected at Tmid = 1400
	Value computed using low-temperature polynomial:  -29.0113435047619
	Value computed using high-temperature polynomial: -28.977686331657146

  gas = Solution(out_name)
C:\Users\Louis\anacondadownload\envs\cantera-env\lib\site-packages\cantera\ck2yaml.py:2344: UserWarning: NasaPoly2::validate: 
For species NEOC5, discontinuity in cp/R detected at Tmid = 1400
	Value computed using low-temperature polynomial:  42.78292259039995
	Value computed using hig

## Verder met code

In [13]:
reac_mech_DLL = 'chem.yaml'
gas_id = 'gas'
gas = ct.Solution(reac_mech_DLL)
print('Gas mechanism contains {} species and {} reactions'.format(gas.n_species, gas.n_reactions))

Gas mechanism contains 239 species and 0 reactions


## Operating Conditions

In [18]:
massflow = [80,85] #np.linspace(75,100,2) #80.4367 #kg/s
diam = 0.5 #m
T_in = [650+273,850+273] #np.linspace(550+273.15,1250+273.15,3)#650 + 273.15 #K
P_in = [1.5e+5,2.0e+5] #np.linspace(1.0e+5,3.5e+5,3)#2.0e+5 #Pa
Y_in = {'C2H6':0.77,'H2O':0.23} #massfractions inlet stream
N_steps = 100 #Aantal integratiestappen

In [32]:
H_input = [0,25e+6,50e+6] #np.linspace(0,50e+6,3) #50000000 #W/m2, discrete heat input
NS = [5,15,30] #np.linspace(5,30,3,dtype=int) #20 # Aantal heat inputs, (aantal passages door de blades vd turboreactor)
H2O_fraction = [0.3,0.5] #np.linspace(0.3,0.5,3)
excel_database = "Database_TEST10.xlsx"
writer1 = pd.ExcelWriter(excel_database, engine='xlsxwriter')
counter = 0
for H in H_input:    
    for N in NS:
        z_prof = np.linspace(0,10.5,100).tolist() #Axiale afstand, m
        hf_prof = [] #Heatflux, W/m2
        for i in range(len(z_prof)):
            if i%(len(z_prof)//N) == 0:
                hf_prof.append(H) #W/m2
            if i%(len(z_prof)//N) != 0:
                hf_prof.append(0)
        length = z_prof[-1] - z_prof[0] #m
        hf = ct.Func1(lambda z: np.interp(z,z_prof,hf_prof))
        y = []
        for i in z_prof:
            y.append(hf(i))
        y.append(0)
        for M in massflow:
            PFR_calc = customPFR(reac_mech_DLL,gas_id,M,diam,CRACKSIM_rates_DLL,energy_type = 'heat-flux-profile',heat_flux = hf, U = None, Tr = None,
                        friction_factors = None)
            for T in T_in:
                for P in P_in:
                    for X in H2O_fraction:
                        PFR_calc.gas.TPY = T, P, {'C2H6':(1.0 - X), 'H2O':X}
                        states_PFR_calc,rates_PFR_calc,enth_PFR_calc = PFR_calc.solve(length,100)
    
                        df_Y = pd.DataFrame(states_PFR_calc.Y, columns = states_PFR_calc.species_names)
                        df_Y.rename(columns = {name: f"Y_{name}" for name in df_Y.columns},inplace = True)
                        df_rr = pd.DataFrame(rates_PFR_calc,columns = states_PFR_calc.species_names)
                        df_rr.rename(columns = {name: f"R_{name}" for name in df_rr.columns}, inplace = True)
                        df_species = pd.concat([df_Y, df_rr], axis = 1)
                        df_species['T [K]'] = states_PFR_calc.T
                        df_species['P [Pa]'] = states_PFR_calc.P
                        df_species['S Energy [J/s/m3]'] = states_PFR_calc.energy_source_term
                        df_species['Mass flow [kg/s]'] = states_PFR_calc.mass_flow
                        df_species['Heat input [W/m^2]'] = y
                        counter += 1
                        sheetname = f"{counter}"
                        df_species.to_excel(writer1, sheet_name = sheetname,index = False)

writer1.close()

InvalidWorksheetName: Excel worksheet name 'T_923P_150000x_0M_80N_5H_25000000' must be <= 31 chars.

## Heatflux profile

In [22]:
# #heatflux = np.genfromtxt('Heatflux.txt')
# z_prof = np.linspace(0,10.5,100).tolist() #Axiale afstand, m
# hf_prof = [] #Heatflux, W/m2
# H_input = 50000000 #W/m2, discrete heat input
# N = 20 # Aantal heat inputs, (aantal passages door de blades vd turboreactor)
# for i in range(len(z_prof)):
#     if i%(len(z_prof)//N) == 0:
#         hf_prof.append(H_input) #W/m2
#     if i%(len(z_prof)//N) != 0:
#         hf_prof.append(0)
# #print(hf_prof)
# #plt.figure()
# #plt.plot(z_prof,hf_prof)
# #plt.show()
# length = z_prof[-1] - z_prof[0] #m
# hf = ct.Func1(lambda z: np.interp(z,z_prof,hf_prof))
# x = []
# y = []
# for i in z_prof:
#     x.append(i)
#     y.append(hf(i))
# y.append(0)

# #plt.figure()
# #plt.plot(x,y)
# #plt.show()

In [48]:
# excelfile = "Database_TEST7.xlsx"
# writer = pd.ExcelWriter(excelfile, engine = 'xlsxwriter')
# for T in np.linspace(550+273.15,1000+273.15,5):
#     PFR_calc = customPFR(reac_mech_DLL,gas_id,80.4367,0.5,CRACKSIM_rates_DLL,energy_type = 'heat-flux-profile',heat_flux = hf, U = None, Tr = None,
#                         friction_factors = None)
#     PFR_calc.gas.TPY = T, 2.0e+5, {'C2H6':0.77,'H2O':0.23}
#     states_PFR_calc,rates_PFR_calc,enth_PFR_calc = PFR_calc.solve(10.5,100)
#     df_Y = pd.DataFrame(states_PFR_calc.Y, columns = states_PFR_calc.species_names)
#     df_Y.rename(columns = {name: f"Y_{name}" for name in df_Y.columns},inplace = True)
#     df_rr = pd.DataFrame(rates_PFR_calc,columns = states_PFR_calc.species_names)
#     df_rr.rename(columns = {name: f"R_{name}" for name in df_rr.columns}, inplace = True)
#     df_species = pd.concat([df_Y, df_rr], axis = 1)
#     df_species['T [K]'] = states_PFR_calc.T
#     df_species['P [Pa]'] = states_PFR_calc.P
#     df_species['S Energy [J/s/m3]'] = states_PFR_calc.energy_source_term
#     df_species['Mass flow [kg/s]'] = states_PFR_calc.mass_flow
#     df_species['Heat input [W/m^2]'] = y
    
#     sheetname = f"T_{int(T)}"
#     df_species.to_excel(writer, sheet_name = sheetname,index = False)
# writer.close()

In [26]:
# print(states_PFR_calc)

In [28]:
# print((enth_PFR_calc[0]))

In [30]:
# print(rates_PFR_calc)